<div style="background: #2c3e50; 
            color: white; 
            padding: 20px; 
            border-radius: 8px;
            margin: 10px 0;">
    <h1 style="margin: 0; font-size: 24px;">
        Étape 1 : Importation des packages
    </h1>
</div>

In [ ]:
import pandas as pd
from datetime import datetime
import json
from bs4 import BeautifulSoup
import re

<div style="background: #2c3e50; 
            color: white; 
            padding: 20px; 
            border-radius: 8px;
            margin: 10px 0;">
    <h1 style="margin: 0; font-size: 24px;">
        Étape 2 : Importation et exploration des données
    </h1>
</div>

In [ ]:
df = pd.read_json("inputs/evenements-publics-openagenda.json")
pd.set_option('display.max_columns', None)
df.head()

<div style="background: #3498db; color: white; padding: 1rem 2rem; border-radius: 5px;">
    <p style="margin: 0;">
        Le dataset contenant un grand nombre de lignes et de colonnes, nous commencerons par sélectionner les variables pertinentes pour le chatbot avant d'analyser le taux de valeurs manquantes.
    </p>
</div>

<div style="background: #27ae60; color: white; padding: 1rem 2rem; border-radius: 5px;">
    <p style="margin: 0;">
    Quelles sont les colonnes disponibles ?    </p>
</div>

In [ ]:
missing_df = (
    df.isna()
      .mean()
      .mul(100)
      .round(2)
      .reset_index()
      .rename(columns={"index": "colonne", 0: "valeur manquante"})
)

print(missing_df)


<div style="background: #3498db; color: white; padding: 1rem 2rem; border-radius: 5px;">
    <p style="margin: 0 0 0.75rem 0; font-weight: 600;">
        Nous avons pas mal de colonnes, mais certaines sont plus importantes que d'autres :
    </p>
    <ul style="margin: 0; padding-left: 1.2rem;">
        <li>title_fr</li>
        <li>description_fr</code></li>
        <li>longdescription_fr</code></li>
        <li>conditions_fr</code></li>
        <li>timings</code></li>
        <li>location_name</code></li>
        <li>location_address</code></li>
        <li>location_insee</code></li>
        <li>location_postalcode</code></li>
        <li>location_city</code></li>
        <li>location_department</code></li>
        <li>location_region</code></li>
        <li>location_countrycode</code></li>
        <li>age_min</code></li>
        <li>age_max</code></li>
        <li>registration</code></li>
    </ul>
</div>


In [ ]:
column_keep = [
    "title_fr",
    "description_fr",
    "longdescription_fr",
    "conditions_fr",
    "timings",
    "location_name",
    "location_address",
    "location_insee",
    "location_postalcode",
    "location_city",
    "location_department",
    "location_region",
    "location_countrycode",
    "age_min",
    "age_max",
    "registration",
    "location_phone",          
    "location_website"
]

In [ ]:
df = df[column_keep]
df.head()

<div style="background: #2c3e50; 
            color: white; 
            padding: 20px; 
            border-radius: 8px;
            margin: 10px 0;">
    <h1 style="margin: 0; font-size: 24px;">
        Étape 3 : Nettoyage des données
    </h1>
</div>

In [ ]:
print(f"Nombre d'événements avant le nettoyage sur les valeurs indispenssable : {len(df)}")

df = df[
    (df['title_fr'].notnull()) & 
    (df['location_address'].notnull()) &
    (df['description_fr'].notnull()) &
    (df['timings'].notnull())
]

print(f"Nombre d'événements après le nettoyage : {len(df)}")

In [ ]:
# Format Age / location_insee / location_postalcode en int
df['location_insee'] = df['location_insee'].astype('Int64')
df['location_postalcode'] = df['location_postalcode'].astype('Int64')
df['age_min'] = df['age_min'].astype('Int64')
df['age_max'] = df['age_max'].astype('Int64')

df['location_phone'] = df['location_phone'].fillna("pas de numéro de téléphone")
df['location_website'] = df['location_website'].fillna("pas de site")

print(df[['location_insee', 'location_postalcode','age_min', 'age_max', 'location_website', 'location_phone']].head())

In [ ]:
df['conditions_fr'] = df['conditions_fr'].fillna("Aucune condition précisée")
df['registration'] = df['registration'].fillna("Pas de lien d'inscription")
print(df[['registration', 'conditions_fr']].head())

In [ ]:
# Nettoyer l'HTML

def remove_html(text):
    if not text or str(text) == 'nan':
        return ""
        
    text_no_html = BeautifulSoup(text, "html.parser").get_text(separator=" ")
    text_clean = re.sub(r'\s+', ' ', text_no_html)
    return text_clean.strip()

df['longdescription_fr' ] = df['longdescription_fr' ].apply(remove_html)

print(df[['longdescription_fr']].head())

In [ ]:
# Nettoyer inscription

def clean_registration(value):
    if not value or str(value) == 'nan':
        return "Entrée libre"
        
    try:
        reg_list = json.loads(value)
        
        infos_contact = []
        
        for item in reg_list:
            type_contact = item.get('type')
            valeur_contact = item.get('value')
            
            if type_contact == 'phone':
                infos_contact.append(f"phone: {valeur_contact}")
            elif type_contact == 'email':
                infos_contact.append(f"email: {valeur_contact}")
            elif type_contact == 'link':
                infos_contact.append(f"link: {valeur_contact}")
            else:
                infos_contact.append(valeur_contact)
        
        if not infos_contact:
            return "Voir description"
            
        return " | ".join(infos_contact)
        
    except Exception:
        return "Voir description"



df['registration'] = df['registration'].apply(clean_registration)

In [ ]:
# Nettoyer timing

def clean_timings(value):
    if not value or str(value) == 'nan':
        return "Horaires non précisés"
    
    try:
        timings_list = json.loads(value)
        
        readable_dates = []
        for t in timings_list:
            start = datetime.fromisoformat(t['begin'])
            end = datetime.fromisoformat(t['end'])
            
            date_str = start.strftime("%d/%m/%Y")
            start_hour = start.strftime("%H:%M")
            end_hour = end.strftime("%H:%M")
            
            readable_dates.append(f"Le {date_str} de {start_hour} à {end_hour}")
            
        return " / ".join(readable_dates)
        
    except (json.JSONDecodeError, ValueError, TypeError):
        return "Horaires invalides"
        
df['timings'] = df['timings'].apply(clean_timings)

In [ ]:
df.head(10)

<div style="background: #2c3e50; 
            color: white; 
            padding: 20px; 
            border-radius: 8px;
            margin: 10px 0;">
    <h1 style="margin: 0; font-size: 24px;">
        Étape 4 : Exportation des données après nettoyage
    </h1>
</div>

In [ ]:
data_list = df.to_dict(orient='records')

with open("outputs/evenements-publics-openagenda-clean.json", 'w', encoding='utf-8') as f: 
    json.dump( data_list, f, ensure_ascii=False,indent=4)

print(f"Fichier nettoyé et sauvegardé dans 'outputs/evenements-publics-openagenda-clean.json'")